# Teste manual do bronze reader

Explora dados em `data/raw/` via [`src/app/lake/bronze/reader.py`](../src/app/lake/bronze/reader.py) (layout Hive: `{dataset}/{partition_key}={value}/part.{ext}`).

**Pré-requisitos**

1. `pip install -e ".[dev]"` na raiz do projeto
2. Bronze já populado: `python run_bronze.py daily` ou `one <dataset>`
3. Kernel no mesmo venv do projeto

Abra na pasta `notebooks/` **ou** na raiz do repositório.

In [1]:
import sys
from datetime import date
from pathlib import Path

import pandas as pd

ROOT = Path.cwd().resolve()
if (ROOT / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT
elif (ROOT.parent / "src" / "config").is_dir():
    PROJECT_ROOT = ROOT.parent
else:
    raise RuntimeError("Execute na raiz do repo ou em notebooks/")

SRC = PROJECT_ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 200)
pd.set_option("display.max_rows", 20)

try:
    from IPython.display import display
except ImportError:
    display = print  # type: ignore[misc, assignment]

from app.config import get_settings
from models.datasets import PIPELINE_NAMES
from app.lake.bronze.partitioning import get_partition_spec
from app.lake.bronze.reader import (
    iter_partitions_in_range,
    list_partition_values,
    partition_values_for_range,
    read_partition,
    read_range,
)

get_settings.cache_clear()
settings = get_settings()

print("PROJECT_ROOT:", PROJECT_ROOT)
print("BRONZE_ROOT:", settings.bronze_root)
print("DATA_START_DATE:", settings.data_start_date)
print("Datasets:", list(PIPELINE_NAMES))

PROJECT_ROOT: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics
BRONZE_ROOT: \\BRZ1WS2478\Brazil_Dept\Mesa_Papel\Chila\projetos\brazil_fixed_income_analytics\data\raw
DATA_START_DATE: 2026-01-01
Datasets: ['mercado_secundario', 'liquidacoes_mercado', 'ajustes_bmf', 'feriados', 'leiloes', 'ipca_indice', 'projecoes']


In [17]:
# Intervalo de leitura (alinhar ao lake / .env)
START_DATE = "2026-05-01"
END_DATE = "2026-05-01"

# Dataset para as células abaixo (troque conforme quiser testar)
DATASET = "projecoes"

print("START_DATE:", START_DATE)
print("END_DATE:", END_DATE)
print("DATASET:", DATASET)


def preview_df(df: pd.DataFrame, title: str, n: int = 8) -> None:
    print("\n" + "=" * 60)
    print(f"{title} | shape={df.shape}")
    print("=" * 60)
    if df is None or df.empty:
        print("(DataFrame vazio)")
        return
    display(df.head(n))
    print("colunas:", list(df.columns))

START_DATE: 2026-05-01
END_DATE: 2026-05-01
DATASET: projecoes


## 1. Inventário de partições no disco

Lista o que existe em `data/raw/` por dataset (sem ler o conteúdo ainda).

In [18]:
rows = []
for name in PIPELINE_NAMES:
    spec = get_partition_spec(name)
    on_disk = list_partition_values(name)
    in_range = partition_values_for_range(name, START_DATE, END_DATE)
    missing_in_range = [v for v in in_range if v not in on_disk]
    rows.append(
        {
            "dataset": name,
            "granularity": spec.granularity,
            "partition_key": spec.partition_key,
            "ext": spec.artifact_ext,
            "partitions_on_disk": len(on_disk),
            "first": on_disk[0] if on_disk else None,
            "last": on_disk[-1] if on_disk else None,
            "candidates_in_range": len(in_range),
            "missing_in_range": len(missing_in_range),
        }
    )

inventory = pd.DataFrame(rows)
display(inventory)

,dataset,granularity,partition_key,ext,partitions_on_disk,first,last,candidates_in_range,missing_in_range
0,mercado_secundario,day,data,json,91,2026-01-02,2026-05-15,1,1
1,liquidacoes_mercado,day,data,parquet,91,2026-01-02,2026-05-15,1,1
2,ajustes_bmf,day,data,parquet,92,2026-01-02,2026-05-18,1,1
3,feriados,snapshot,snapshot,parquet,1,1,1,1,0
4,leiloes,day,data,json,38,2026-01-06,2026-05-14,1,1
5,ipca_indice,month,reference_month,parquet,4,2026-01-01,2026-04-01,1,1
6,projecoes,month,reference_month,json,5,2026-01-01,2026-05-01,1,0


## 2. `read_range` — um dataset no intervalo

Por padrão lê só partições **existentes** no disco dentro de `[START_DATE, END_DATE]` (granularidade dia/mês/snapshot conforme o spec).

In [19]:
spec = get_partition_spec(DATASET)
print(f"{DATASET}: {spec.granularity} | {spec.partition_key}=... | .{spec.artifact_ext}")

df_range = read_range(DATASET, START_DATE, END_DATE)
preview_df(df_range, f"read_range({DATASET!r}, {START_DATE}, {END_DATE})")

projecoes: month | reference_month=... | .json

read_range('projecoes', 2026-05-01, 2026-05-01) | shape=(4, 6)


,indice,tipo_projecao,data_coleta,mes_referencia,variacao_projetada,data_validade
0,IPCA,PROJEÇÕES PARA O MÊS CORRENTE,2026-05-12,05/2026,0.50,2026-05-18
1,IPCA,PROJEÇÕES PARA O MÊS POSTERIOR,2026-05-12,06/2026,0.33,NaN
2,IGP-M,PROJEÇÕES PARA O MÊS CORRENTE,2026-05-12,05/2026,0.70,2026-05-13
3,IGP-M,PROJEÇÕES PARA O MÊS POSTERIOR,2026-05-12,06/2026,0.40,NaN


colunas: ['indice', 'tipo_projecao', 'data_coleta', 'mes_referencia', 'variacao_projetada', 'data_validade']


## 3. `read_partition` — uma partição específica

Útil para inspecionar um único `data=YYYY-MM-DD` ou `reference_month=YYYY-MM-01`.

In [ ]:
parts = list_partition_values(DATASET)
if not parts:
    print(f"Nenhuma partição em disco para {DATASET!r}. Rode: python run_bronze.py one {DATASET}")
else:
    PARTITION_VALUE = parts[-1]  # última disponível; troque manualmente se quiser
    print("PARTITION_VALUE:", PARTITION_VALUE)
    df_one = read_partition(DATASET, PARTITION_VALUE)
    preview_df(df_one, f"read_partition({DATASET!r}, {PARTITION_VALUE!r})")

## 4. Coluna de partição (`add_partition_column=True`)

Inclui `_partition_{partition_key}` para saber a origem de cada linha após concat.

In [ ]:
df_tagged = read_range(
    DATASET,
    START_DATE,
    END_DATE,
    add_partition_column=True,
)
preview_df(df_tagged, f"read_range + add_partition_column ({DATASET})")

## 5. `iter_partitions_in_range` — partição a partição

Menos memória que `read_range` quando há muitas partições.

In [ ]:
MAX_SHOW = 3
for i, ref in enumerate(iter_partitions_in_range(DATASET, START_DATE, END_DATE)):
    if i >= MAX_SHOW:
        print(f"... (mostrando só {MAX_SHOW} partições)")
        break
    print(f"\n--- {ref.partition_key}={ref.partition_value} ---")
    print("path:", ref.path)
    chunk = read_partition(ref.dataset, ref.partition_value)
    preview_df(chunk, ref.partition_value, n=3)

## 6. Exemplos rápidos por tipo de dataset

Descomente o bloco que quiser testar.

In [ ]:
# Diário (JSON)
# preview_df(read_range("mercado_secundario", START_DATE, END_DATE), "mercado_secundario")
# preview_df(read_range("leiloes", START_DATE, END_DATE), "leiloes")

# Diário (Parquet)
# preview_df(read_range("liquidacoes_mercado", START_DATE, END_DATE), "liquidacoes_mercado")
# preview_df(read_range("ajustes_bmf", START_DATE, END_DATE), "ajustes_bmf")

# Mensal
# preview_df(read_range("ipca_indice", START_DATE, END_DATE), "ipca_indice")
# preview_df(read_range("projecoes", START_DATE, END_DATE), "projecoes")

# Snapshot
# preview_df(read_range("feriados", START_DATE, END_DATE), "feriados")

print("Descomente as linhas acima ou altere DATASET na célula de configuração.")

## Referência da API

| Função | Descrição |
|--------|----------|
| `list_partition_values(dataset)` | Partições com arquivo no disco |
| `partition_values_for_range(dataset, start, end)` | Candidatos no intervalo (dia útil / mês / snapshot) |
| `read_partition(dataset, value)` | Uma partição → DataFrame |
| `read_range(dataset, start, end)` | Intervalo (default: só existentes) |
| `iter_partitions_in_range(dataset, start, end)` | Itera `BronzePartitionRef` com `.path` |

Parâmetros úteis em `read_range` / `read_partitions`: `skip_missing`, `only_existing`, `add_partition_column`.